# Notebook 01 · Construcción de índices competitivos `player_based_v1`

Este notebook explica cómo se construyen los índices de equipo a partir de jugadores individuales.

Flujo:

1. Seleccionar jugadores.
2. Asignar pesos por minutos esperados.
3. Agregar atributos por línea.
4. Calcular ataque, medio, defensa, portería y banquillo.
5. Construir el **WCPI**.

> Los ratings utilizados son estimaciones modeladas, no datos oficiales.

## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "pmcw").exists():
    sys.path.insert(0, str(repo_root))

from pmcw.ratings import aggregate_player_based

## 2. Dataset de ejemplo

En el proyecto completo se trabaja con 16 jugadores por selección. Aquí usamos una muestra suficientemente rica para seguir el cálculo.

In [ ]:
players = pd.DataFrame([
    ["ESP","Unai Simón","GK","starter",1.00,20,35,45,89,86],
    ["ESP","Carvajal","DEF","starter",0.95,74,78,84,10,83],
    ["ESP","Le Normand","DEF","starter",0.95,58,68,86,10,81],
    ["ESP","Laporte","DEF","starter",0.95,61,72,87,10,83],
    ["ESP","Cucurella","DEF","starter",0.90,72,77,81,10,80],
    ["ESP","Rodri","MID","starter",1.00,78,92,88,10,91],
    ["ESP","Pedri","MID","starter",0.95,82,91,72,10,88],
    ["ESP","Fabián Ruiz","MID","starter",0.90,80,86,76,10,85],
    ["ESP","Lamine Yamal","ATT","starter",0.95,90,84,55,10,89],
    ["ESP","Nico Williams","ATT","starter",0.90,87,80,58,10,86],
    ["ESP","Morata","ATT","starter",0.90,85,74,60,10,84],
    ["ESP","Dani Olmo","MID","impact_sub",0.45,86,85,66,10,85],
    ["ESP","Oyarzabal","ATT","impact_sub",0.40,84,78,62,10,82],
    ["ESP","Merino","MID","rotation_sub",0.35,75,82,80,10,82],

    ["AUT","Pentz","GK","starter",1.00,15,30,42,81,79],
    ["AUT","Posch","DEF","starter",0.95,65,69,80,10,78],
    ["AUT","Lienhart","DEF","starter",0.95,54,65,82,10,79],
    ["AUT","Danso","DEF","starter",0.95,50,62,84,10,80],
    ["AUT","Mwene","DEF","starter",0.90,68,72,77,10,76],
    ["AUT","Laimer","MID","starter",1.00,74,84,82,10,83],
    ["AUT","Seiwald","MID","starter",0.95,68,81,80,10,80],
    ["AUT","Sabitzer","MID","starter",0.95,82,84,72,10,83],
    ["AUT","Baumgartner","MID","starter",0.90,81,82,69,10,81],
    ["AUT","Gregoritsch","ATT","starter",0.90,80,70,58,10,79],
    ["AUT","Arnautović","ATT","starter",0.85,83,72,56,10,80],
    ["AUT","Wimmer","ATT","impact_sub",0.40,79,74,60,10,78],
    ["AUT","Grillitsch","MID","rotation_sub",0.35,67,78,77,10,77],
    ["AUT","Prass","MID","rotation_sub",0.35,72,76,73,10,76],
], columns=[
    "team_id","player_name","position_group","expected_role",
    "minutes_weight","attack","midfield","defense","goalkeeper","overall"
])

players.head()

## 3. Pesos por rol

In [ ]:
minutes_summary = (
    players.groupby(["team_id","expected_role"])["minutes_weight"]
    .sum()
    .unstack(fill_value=0)
)
minutes_summary

In [ ]:
ax = minutes_summary.plot(kind="bar", figsize=(9,5))
ax.set_title("Peso acumulado de minutos por rol")
ax.set_xlabel("Selección")
ax.set_ylabel("Suma de minutes_weight")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Agregación de ratings

Las fórmulas combinan el rendimiento de las distintas líneas. El objetivo es evitar que un índice dependa de un único jugador.

In [ ]:
team_indices = {}
for team_id, team_df in players.groupby("team_id"):
    team_indices[team_id] = aggregate_player_based(team_df.to_dict("records"))

indices_df = pd.DataFrame(team_indices).T.round(2)
indices_df

## 5. Comparación visual

In [ ]:
comparison = indices_df[
    ["attack_index","midfield_index","defense_index",
     "goalkeeper_index","bench_index","WCPI"]
]

ax = comparison.T.plot(kind="bar", figsize=(11,6))
ax.set_title("Comparación de índices competitivos")
ax.set_xlabel("Índice")
ax.set_ylabel("Rating")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 6. Interpretación

- `attack_index` refleja capacidad ofensiva agregada.
- `midfield_index` aproxima control y creación.
- `defense_index` combina estructura defensiva y apoyo del resto de líneas.
- `goalkeeper_index` representa el nivel del portero.
- `bench_index` mide profundidad.
- `WCPI` resume la fuerza competitiva global.

La transparencia no depende solo del resultado final, sino de mostrar cómo se construye cada input.